# Ours full scan (all runs)

This notebook scans **all** `report_normal_1day.pkl` under `/home/dgu/fin/01_15_new_qlib/runs` and computes AR/IR/MDD/CR.

Warning: this can be slow (tens of thousands of files).

In [45]:
from __future__ import annotations
from pathlib import Path
import pandas as pd
import numpy as np
import time


In [46]:
ROOT = Path('/home/dgu/fin/01_15_new_qlib/runs')
OUT_PATH = Path('/home/dgu/fin/01_15_new_qlib/analysis/ours_perf_full.csv')


In [47]:
import json


def _region_from_run_config(report_pkl: Path):
    # expected: .../runs/<run_id>/run_config.json
    for parent in report_pkl.parents:
        cand = parent / 'run_config.json'
        if cand.exists():
            try:
                obj = json.loads(cand.read_text(encoding='utf-8'))
                q = (obj.get('config') or {}).get('qlib') or {}
                region = q.get('region')
                market = q.get('qlib_market') or q.get('market')
                benchmark = q.get('benchmark')
                provider_uri = q.get('provider_uri')
                return {
                    'region': region,
                    'market': market,
                    'benchmark': benchmark,
                    'provider_uri': provider_uri,
                    'conf_path': str(cand),
                }
            except Exception:
                return {}
    return {}


def _run_id_from_path(report_pkl: Path):
    try:
        parts = report_pkl.parts
        i = parts.index('runs')
        return parts[i + 1]
    except Exception:
        return None


In [48]:
def _compute_curves(report_df: pd.DataFrame) -> pd.DataFrame:
    df = report_df.copy()
    if not isinstance(df.index, pd.DatetimeIndex):
        try:
            df.index = pd.to_datetime(df.index)
        except Exception:
            pass
    df = df.sort_index()
    idx = df.index
    gross = pd.Series(df.get('return', 0.0), index=idx, dtype=float).fillna(0.0).rename('gross')
    cost = pd.Series(df.get('cost', 0.0), index=idx, dtype=float).fillna(0.0).rename('cost')
    net = (gross - cost).rename('net')
    bench = None
    if 'bench' in df.columns:
        bench = pd.Series(df.get('bench', 0.0), index=idx, dtype=float).fillna(0.0).rename('bench')
    nav_gross = (1.0 + gross).cumprod().rename('nav_gross')
    nav_net = (1.0 + net).cumprod().rename('nav_net')
    out = pd.concat([gross, cost, net, nav_gross, nav_net], axis=1)
    if bench is not None:
        nav_bench = (1.0 + bench).cumprod().rename('nav_bench')
        out = pd.concat([out, bench, nav_bench], axis=1)
    return out


def _calc_perf_from_report(report_df: pd.DataFrame) -> dict:
    curves = _compute_curves(report_df)
    bench = curves['bench'] if 'bench' in curves.columns else 0.0
    daily = (curves['net'] - bench).dropna()
    if len(daily) == 0:
        return {'n_days': 0, 'CR': np.nan, 'AR': np.nan, 'IR': np.nan, 'MDD': np.nan, 'mean': np.nan, 'std': np.nan}
    n_days = int(len(daily))
    cr = float((1.0 + daily).prod() - 1.0)
    ar = float((1.0 + cr) ** (252.0 / n_days) - 1.0) if n_days > 0 else np.nan
    mean = float(daily.mean())
    std = float(daily.std(ddof=0))
    ir = float((mean / std) * np.sqrt(252.0)) if std > 0 else np.nan
    rel_nav = (1.0 + daily).cumprod()
    mdd = float((rel_nav / rel_nav.cummax() - 1.0).min())
    return {'n_days': n_days, 'CR': cr, 'AR': ar, 'IR': ir, 'MDD': mdd, 'mean': mean, 'std': std}


In [49]:
# Scan all reports
report_paths = list(ROOT.rglob('report_normal_1day.pkl'))
print('reports:', len(report_paths))


reports: 16694


In [50]:
rows = []
start = time.time()
for i, p in enumerate(report_paths, 1):
    meta = _region_from_run_config(p)
    run_id = _run_id_from_path(p)
    try:
        df = pd.read_pickle(p)
    except Exception as e:
        row = {'path': str(p), 'run_id': run_id}
        row.update(meta)
        row['error'] = f'{type(e).__name__}: {e}'
        rows.append(row)
        continue
    try:
        perf = _calc_perf_from_report(df)
    except Exception as e:
        row = {'path': str(p), 'run_id': run_id}
        row.update(meta)
        row['error'] = f'calc_failed: {type(e).__name__}: {e}'
        rows.append(row)
        continue
    row = {'path': str(p), 'run_id': run_id}
    row.update(meta)
    row.update(perf)
    rows.append(row)

    if i % 500 == 0:
        print(f'processed {i}/{len(report_paths)}')

print('elapsed_s:', round(time.time() - start, 1))


processed 500/16694
processed 1000/16694
processed 4500/16694
processed 5500/16694
processed 7000/16694
processed 8500/16694
processed 9500/16694
processed 11000/16694
processed 16000/16694
elapsed_s: 530.5


In [51]:
out_df = pd.DataFrame(rows)
# keep valid rows with metrics
out_df.to_csv(OUT_PATH, index=False)
print('wrote:', OUT_PATH)

valid = out_df[pd.notna(out_df['IR']) & pd.notna(out_df['MDD'])].copy()
valid['abs_mdd'] = valid['MDD'].abs()
valid['score'] = valid['IR'] - valid['abs_mdd']
print('Top 10 by IR - abs(MDD):')
print(valid.sort_values(['score','IR'], ascending=False).head(10)[['path','IR','MDD','AR','CR','n_days','score']].to_string(index=False))


wrote: /home/dgu/fin/01_15_new_qlib/analysis/ours_perf_full.csv
Top 10 by IR - abs(MDD):
                                                                                                      path       IR       MDD       AR        CR  n_days    score
/home/dgu/fin/01_15_new_qlib/runs/20260205_033043/qlib_artifacts/iter_3/combo_55/is/report_normal_1day.pkl 1.635822 -0.348572 0.710275 21.498437  1462.0 1.287251
/home/dgu/fin/01_15_new_qlib/runs/20260203_063015/qlib_artifacts/iter_5/combo_13/is/report_normal_1day.pkl 1.499126 -0.276907 0.452605  7.723943  1462.0 1.222219
/home/dgu/fin/01_15_new_qlib/runs/20260206_142717/qlib_artifacts/iter_5/combo_83/is/report_normal_1day.pkl 1.363707 -0.239548 0.659766 17.906937  1462.0 1.124159
/home/dgu/fin/01_15_new_qlib/runs/20260205_033043/qlib_artifacts/iter_3/combo_63/is/report_normal_1day.pkl 1.440706 -0.332712 0.612856 15.009701  1462.0 1.107994
 /home/dgu/fin/01_15_new_qlib/runs/20260206_144300/qlib_artifacts/iter_4/combo_1/is/report_normal_1da

In [52]:
valid.sort_values(['score','IR'], ascending=False).groupby(['path'])['IR'].mean().sort_values(ascending=False)

path
/home/dgu/fin/01_15_new_qlib/runs/20260205_033043/qlib_artifacts/iter_3/combo_55/is/report_normal_1day.pkl              1.635822
/home/dgu/fin/01_15_new_qlib/runs/20260203_063015/qlib_artifacts/iter_5/combo_13/is/report_normal_1day.pkl              1.499126
/home/dgu/fin/01_15_new_qlib/runs/20260206_144300/qlib_artifacts/iter_4/combo_1/is/report_normal_1day.pkl               1.452470
/home/dgu/fin/01_15_new_qlib/runs/20260205_033043/qlib_artifacts/iter_3/combo_63/is/report_normal_1day.pkl              1.440706
/home/dgu/fin/01_15_new_qlib/runs/20260206_142717/qlib_artifacts/iter_5/combo_89/is/report_normal_1day.pkl              1.433383
                                                                                                                          ...   
/home/dgu/fin/01_15_new_qlib/runs/20260127_211022/qlib_artifacts/iter_2/combo_2/fixed_q85/is/report_normal_1day.pkl    -3.196098
/home/dgu/fin/01_15_new_qlib/runs/20260127_211022/qlib_artifacts/iter_2/combo_1/oos/report_n

In [53]:
valid.query('region=="cn"').sort_values(['score','IR'], ascending=False).query('MDD>-0.3').head(60)

,path,run_id,region,market,benchmark,provider_uri,conf_path,n_days,CR,AR,IR,MDD,mean,std,error,abs_mdd,score
7353,/home/dgu/fin/01_15_new_qlib/runs/20260203_063...,20260203_063015,cn,csi500,SH000905,~/.qlib/qlib_data/cn_data,/home/dgu/fin/01_15_new_qlib/runs/20260203_063...,1462.0,7.723943,0.452605,1.499126,-0.276907,0.001629,0.017249,NaN,0.276907,1.222219
3837,/home/dgu/fin/01_15_new_qlib/runs/20260206_142...,20260206_142717,cn,csi500,SH000905,~/.qlib/qlib_data/cn_data,/home/dgu/fin/01_15_new_qlib/runs/20260206_142...,1462.0,17.906937,0.659766,1.363707,-0.239548,0.002389,0.027811,NaN,0.239548,1.124159
3749,/home/dgu/fin/01_15_new_qlib/runs/20260206_142...,20260206_142717,cn,csi500,SH000905,~/.qlib/qlib_data/cn_data,/home/dgu/fin/01_15_new_qlib/runs/20260206_142...,1462.0,22.523978,0.723466,1.379945,-0.290888,0.002585,0.029734,NaN,0.290888,1.089057
7359,/home/dgu/fin/01_15_new_qlib/runs/20260203_063...,20260203_063015,cn,csi500,SH000905,~/.qlib/qlib_data/cn_data,/home/dgu/fin/01_15_new_qlib/runs/20260203_063...,1462.0,7.624361,0.449733,1.363295,-0.276907,0.001658,0.019309,NaN,0.276907,1.086388
14749,/home/dgu/fin/01_15_new_qlib/runs/20260206_144...,20260206_144300,cn,csi500,SH000905,~/.qlib/qlib_data/cn_data,/home/dgu/fin/01_15_new_qlib/runs/20260206_144...,1462.0,12.034426,0.556699,1.202783,-0.210790,0.002163,0.028542,NaN,0.210790,0.991992
8703,/home/dgu/fin/01_15_new_qlib/runs/20260204_181...,20260204_181138,cn,csi500,SH000905,~/.qlib/qlib_data/cn_data,/home/dgu/fin/01_15_new_qlib/runs/20260204_181...,1462.0,5.087678,0.365253,1.254009,-0.281394,0.001389,0.017578,NaN,0.281394,0.972615
11023,/home/dgu/fin/01_15_new_qlib/runs/20260205_093...,20260205_093103,cn,csi500,SH000905,~/.qlib/qlib_data/cn_data,/home/dgu/fin/01_15_new_qlib/runs/20260205_093...,1462.0,4.506557,0.341847,1.145854,-0.285876,0.001336,0.018515,NaN,0.285876,0.859979
14717,/home/dgu/fin/01_15_new_qlib/runs/20260206_144...,20260206_144300,cn,csi500,SH000905,~/.qlib/qlib_data/cn_data,/home/dgu/fin/01_15_new_qlib/runs/20260206_144...,1462.0,9.176353,0.491679,1.134036,-0.295745,0.001962,0.027463,NaN,0.295745,0.838291
3413,/home/dgu/fin/01_15_new_qlib/runs/20260206_142...,20260206_142717,cn,csi500,SH000905,~/.qlib/qlib_data/cn_data,/home/dgu/fin/01_15_new_qlib/runs/20260206_142...,1462.0,8.055795,0.461983,1.071386,-0.241059,0.001900,0.028153,NaN,0.241059,0.830327
7311,/home/dgu/fin/01_15_new_qlib/runs/20260203_063...,20260203_063015,cn,csi500,SH000905,~/.qlib/qlib_data/cn_data,/home/dgu/fin/01_15_new_qlib/runs/20260203_063...,1462.0,3.890645,0.314691,1.074636,-0.276907,0.001255,0.018542,NaN,0.276907,0.797728


In [54]:
valid.query('region=="us"').sort_values(['score','IR'], ascending=False).values[:3]

array([['/home/dgu/fin/01_15_new_qlib/runs/20260126_204618/qlib_artifacts/iter_1/combo_2/fixed_q90/oos/report_normal_1day.pkl',
        '20260126_204618', 'us', 'sp500', '^GSPC',
        '~/.qlib/qlib_data/us_sp500_qlib',
        '/home/dgu/fin/01_15_new_qlib/runs/20260126_204618/run_config.json',
        1255.0, 0.8490243466502769, 0.13136106631839173,
        0.7276582005787168, -0.21279798654291504, 0.0005661900084734712,
        0.012351936302282963, nan, 0.21279798654291504,
        0.5148602140358017],
       ['/home/dgu/fin/01_15_new_qlib/runs/20260126_204618/qlib_artifacts/iter_1/combo_1/fixed_q85/oos/report_normal_1day.pkl',
        '20260126_204618', 'us', 'sp500', '^GSPC',
        '~/.qlib/qlib_data/us_sp500_qlib',
        '/home/dgu/fin/01_15_new_qlib/runs/20260126_204618/run_config.json',
        1255.0, 0.8482879657021192, 0.1312705791320612,
        0.6854442365215023, -0.21249178825985582, 0.000579273603834284,
        0.013415654973662521, nan, 0.21249178825985582,
   

In [55]:
# ==== PAIRWISE_COMPARISON_CONFIG ====
from pathlib import Path
import pandas as pd
import numpy as np

COMPARE_CONFIG = {
    # Paths to scan
    "Ours": {
        "roots": [Path('/home/dgu/fin/01_15_new_qlib/runs')],
        "patterns": ["**/report_normal_1day.pkl"],
        "kind": "pkl",
        "filter_contains": "oos",
    },
    "AlphaAgent": {
        "roots": [Path('/home/dgu/fin/AlphaAgent/results')],
        "patterns": ["**/report_normal_1day.pkl"],
        "kind": "pkl",
        "filter_contains": None,
    },
    "AlphaForge": {
        "roots": [Path('/home/dgu/fin/others/02_agent/AlphaForge/out'), Path('/home/dgu/fin/others/02_agent/AlphaForge/out_cn')],
        "patterns": ["**/timeseries_*_*.csv"],
        "kind": "csv",
        "filter_contains": None,
    },
    "R&D-Agent": {
        "roots": [Path('/home/dgu/fin/01_15_new_qlib copy')],
        "patterns": ["**/report_normal_1day.pkl"],
        "kind": "pkl",
        "filter_contains": None,
    },
}

# If you have specific paths, list them here (optional):
EXTRA_PATHS = {
    # "R&D-Agent": ["/home/dgu/fin/01_15_new_qlib copy/2026-02-03_06-31-06-947364.pkl"],
}

# Limit to avoid huge combinatorics (tune as needed)
MAX_PER_MODEL = {
    "Ours": 200,          # top N by IR after initial metric calc
    "AlphaAgent": 200,
    "AlphaForge": 200,
    "R&D-Agent": 200,
}

# Minimum overlap days for fair comparison
MIN_OVERLAP_DAYS = 252

In [56]:
# ==== Helpers: region + load daily excess ====
import json

TRADING_DAYS = 252

def _infer_region_from_path(p: Path):
    s = str(p).lower()
    if 'csi' in s or 'cn' in s:
        return 'cn'
    if 'sp500' in s or 'snp' in s or 'us' in s:
        return 'us'
    return None


def _region_from_run_config(p: Path):
    for parent in p.parents:
        cand = parent / 'run_config.json'
        if cand.exists():
            try:
                obj = json.loads(cand.read_text(encoding='utf-8'))
                q = (obj.get('config') or {}).get('qlib') or {}
                r = q.get('region')
                if r in {'cn','us'}:
                    return r
            except Exception:
                return None
    return None


def _compute_curves_from_report(df: pd.DataFrame):
    if not isinstance(df.index, pd.DatetimeIndex):
        try:
            df.index = pd.to_datetime(df.index)
        except Exception:
            pass
    df = df.sort_index()
    idx = df.index
    gross = pd.Series(df.get('return', 0.0), index=idx, dtype=float).fillna(0.0)
    cost = pd.Series(df.get('cost', 0.0), index=idx, dtype=float).fillna(0.0)
    net = (gross - cost)
    bench = pd.Series(df.get('bench', 0.0), index=idx, dtype=float).fillna(0.0) if 'bench' in df.columns else pd.Series(0.0, index=idx)
    return net, bench


def _load_daily_excess(path: Path):
    # CSV: use active_ret or strategy-benchmark
    if str(path).lower().endswith('.csv'):
        df = pd.read_csv(path)
        if 'date' in df.columns:
            idx = pd.to_datetime(df['date'], errors='coerce')
        else:
            idx = pd.RangeIndex(len(df))
        if 'active_ret' in df.columns:
            daily = pd.Series(df['active_ret'].astype(float), index=idx)
        else:
            strat = df['strategy_ret'].astype(float) if 'strategy_ret' in df.columns else None
            bench = df['benchmark_ret'].astype(float) if 'benchmark_ret' in df.columns else None
            if strat is None:
                raise ValueError(f"CSV missing strategy_ret: {path}")
            if bench is None:
                bench = 0.0
            daily = pd.Series(strat - bench, index=idx)
        return daily.replace([np.inf, -np.inf], np.nan).fillna(0.0)

    # PKL: report_normal_1day.pkl
    df = pd.read_pickle(path)
    net, bench = _compute_curves_from_report(df)
    daily = (net - bench).astype(float).fillna(0.0)
    return daily


def _metrics_from_daily(daily: pd.Series):
    daily = daily.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    n = int(len(daily))
    if n == 0:
        return None
    cr = float((1.0 + daily).prod() - 1.0)
    ar = float((1.0 + cr) ** (TRADING_DAYS / n) - 1.0) if cr > -1.0 else np.nan
    mean = float(daily.mean())
    std = float(daily.std(ddof=0))
    ir = float((mean / std) * np.sqrt(TRADING_DAYS)) if std > 0 else np.nan
    nav = (1.0 + daily).cumprod()
    mdd = float((nav / nav.cummax() - 1.0).min())
    return {
        'n_days': n, 'CR': cr, 'AR': ar, 'IR': ir, 'MDD': mdd, 'mean': mean, 'std': std,
        'start': str(daily.index.min()), 'end': str(daily.index.max())
    }

In [57]:
# ==== Scan candidates ====
from collections import defaultdict

candidates = defaultdict(list)
for model, cfg in COMPARE_CONFIG.items():
    paths = []
    for root in cfg['roots']:
        if not root.exists():
            continue
        for pat in cfg['patterns']:
            paths += list(root.rglob(pat))
    # add explicit paths
    for p in EXTRA_PATHS.get(model, []):
        paths.append(Path(p))

    # filter
    if cfg.get('filter_contains'):
        key = cfg['filter_contains']
        paths = [p for p in paths if key in str(p)]

    # compute metrics for ranking (full period)
    rows = []
    for p in paths:
        try:
            daily = _load_daily_excess(p)
            met = _metrics_from_daily(daily)
            if met is None:
                continue
            region = _region_from_run_config(p) or _infer_region_from_path(p)
            rows.append({
                'model': model,
                'path': str(p),
                'region': region,
                **met,
            })
        except Exception:
            continue

    df = pd.DataFrame(rows)
    if len(df) == 0:
        candidates[model] = []
        continue

    # keep top by IR to limit
    df = df.sort_values(['IR','AR'], ascending=False, na_position='last')
    limit = MAX_PER_MODEL.get(model, 200)
    df = df.head(limit)
    candidates[model] = df.to_dict('records')

print({k: len(v) for k,v in candidates.items()})

{'Ours': 200, 'AlphaAgent': 49, 'AlphaForge': 6, 'R&D-Agent': 4}


In [58]:
# ==== Pairwise comparison: Ours vs (AlphaAgent/AlphaForge/R&D-Agent) on common period & region ====

pair_rows = []
ours_list = candidates.get('Ours', [])

for other_name in ['AlphaAgent','AlphaForge','R&D-Agent']:
    other_list = candidates.get(other_name, [])
    for o in ours_list:
        for x in other_list:
            # region match required
            if o.get('region') and x.get('region') and o['region'] != x['region']:
                continue

            try:
                d1 = _load_daily_excess(Path(o['path']))
                d2 = _load_daily_excess(Path(x['path']))
            except Exception:
                continue

            # align on common index
            common_idx = d1.index.intersection(d2.index)
            if len(common_idx) < MIN_OVERLAP_DAYS:
                continue
            d1c = d1.loc[common_idx]
            d2c = d2.loc[common_idx]

            m1 = _metrics_from_daily(d1c)
            m2 = _metrics_from_daily(d2c)
            if not m1 or not m2:
                continue

            pair_rows.append({
                'region': o.get('region') or x.get('region'),
                'overlap_days': len(common_idx),
                'ours_path': o['path'],
                'other_model': other_name,
                'other_path': x['path'],
                'ours_IR': m1['IR'],
                'ours_MDD': m1['MDD'],
                'ours_AR': m1['AR'],
                'other_IR': m2['IR'],
                'other_MDD': m2['MDD'],
                'other_AR': m2['AR'],
                'delta_IR': m1['IR'] - m2['IR'],
                'delta_MDD': m1['MDD'] - m2['MDD'],
            })

pair_df = pd.DataFrame(pair_rows)
print('pairs', len(pair_df))

# sort by ours IR high & ours MDD close to 0 (higher)
if len(pair_df) > 0:
    pair_df = pair_df.sort_values(['ours_IR','ours_MDD'], ascending=[False, False])

pair_df.head(50)

pairs 10326


,region,overlap_days,ours_path,other_model,other_path,ours_IR,ours_MDD,ours_AR,other_IR,other_MDD,other_AR,delta_IR,delta_MDD
9803,cn,1148,/home/dgu/fin/01_15_new_qlib/runs/20260206_144...,AlphaForge,/home/dgu/fin/others/02_agent/AlphaForge/out/b...,1.259960,-0.257292,0.358712,NaN,0.000000,0.000000,NaN,-0.257292
49,cn,1165,/home/dgu/fin/01_15_new_qlib/runs/20260206_144...,AlphaAgent,/home/dgu/fin/AlphaAgent/results/2026-02-03_05...,1.186520,-0.288534,0.330547,1.122064,-0.295280,0.678649,0.064456,0.006746
9801,cn,1179,/home/dgu/fin/01_15_new_qlib/runs/20260206_142...,AlphaForge,/home/dgu/fin/others/02_agent/AlphaForge/out/t...,1.084120,-0.326010,0.318616,NaN,0.000000,0.000000,NaN,-0.326010
9802,cn,1179,/home/dgu/fin/01_15_new_qlib/runs/20260206_142...,AlphaForge,/home/dgu/fin/others/02_agent/AlphaForge/out_c...,1.084120,-0.326010,0.318616,NaN,0.000000,0.000000,NaN,-0.326010
9800,cn,1148,/home/dgu/fin/01_15_new_qlib/runs/20260206_142...,AlphaForge,/home/dgu/fin/others/02_agent/AlphaForge/out/b...,1.076344,-0.317380,0.316072,NaN,0.000000,0.000000,NaN,-0.317380
0,cn,1165,/home/dgu/fin/01_15_new_qlib/runs/20260206_142...,AlphaAgent,/home/dgu/fin/AlphaAgent/results/2026-02-03_05...,1.024433,-0.317380,0.299985,1.122064,-0.295280,0.678649,-0.097631,-0.022100
9806,cn,1148,/home/dgu/fin/01_15_new_qlib/runs/20260206_142...,AlphaForge,/home/dgu/fin/others/02_agent/AlphaForge/out/b...,1.007931,-0.295605,0.286955,NaN,0.000000,0.000000,NaN,-0.295605
9804,cn,1179,/home/dgu/fin/01_15_new_qlib/runs/20260206_144...,AlphaForge,/home/dgu/fin/others/02_agent/AlphaForge/out/t...,0.992230,-0.262459,0.265656,NaN,0.000000,0.000000,NaN,-0.262459
9805,cn,1179,/home/dgu/fin/01_15_new_qlib/runs/20260206_144...,AlphaForge,/home/dgu/fin/others/02_agent/AlphaForge/out_c...,0.992230,-0.262459,0.265656,NaN,0.000000,0.000000,NaN,-0.262459
1,cn,1209,/home/dgu/fin/01_15_new_qlib/runs/20260206_142...,AlphaAgent,/home/dgu/fin/AlphaAgent/results/2026-02-03_10...,0.984861,-0.326010,0.285148,0.166779,-0.145056,0.009317,0.818082,-0.180954


In [59]:
# Save results
if 'pair_df' in globals():
    out_path = Path('/home/dgu/fin/01_15_new_qlib/analysis/ours_pairwise_compare.csv')
    pair_df.to_csv(out_path, index=False)
    print('wrote', out_path)

wrote /home/dgu/fin/01_15_new_qlib/analysis/ours_pairwise_compare.csv
